<a href="https://colab.research.google.com/github/abhinavnautiyalDS/Finwise-GenAI-Assistant/blob/main/finwise-genai-capstone/task-05-sql-qa/task_5_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installing Required packages

In [ ]:
%pip install langchain-community langchain-google-genai sqlalchemy
%pip install ipython-sql

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.77
    Uninstalling langchain-core-0.3.77:
      Successfully uninstalled langchain-core-0.3.77
  Attempting uninstall: google-ai-generativelanguage
    Found existing installation: google-ai-generativelanguage 0.6.15
    Uninstalling google-ai-generativelanguage-0.6.15:
     

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 30.7 MB/s eta 0:00:00


Importing libraries

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import google.generativeai as genai
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain.agents import AgentExecutor, create_sql_agent

Creating a connection to the SQLite database

In [ ]:
conn = sqlite3.connect('financial_data.db')
cursor = conn.cursor()

# Create 'clients' table
cursor.execute('''
CREATE TABLE IF NOT EXISTS clients (
    client_id INTEGER PRIMARY KEY,
    name TEXT,
    age INTEGER,
    risk_profile TEXT,
    portfolio_value REAL
)
''')

Create Tables

In [ ]:


# Create 'investments' table
cursor.execute('''
CREATE TABLE IF NOT EXISTS investments (
    investment_id INTEGER PRIMARY KEY,
    client_id INTEGER,
    fund_name TEXT,
    amount_invested REAL,
    date TEXT,
    FOREIGN KEY (client_id) REFERENCES clients(client_id)
)
''')

# --- Generate Sample Data ---

# Client Data
num_clients = 35
client_data = {
    'client_id': range(1, num_clients + 1),
    'name': [f'Client {i}' for i in range(1, num_clients + 1)],
    'age': np.random.randint(25, 70, num_clients),
    'risk_profile': np.random.choice(['Low', 'Medium', 'High'], num_clients, p=[0.4, 0.3, 0.3]),
    'portfolio_value': np.random.uniform(50000, 5000000, num_clients).round(2)
}
clients_df = pd.DataFrame(client_data)
clients_df['portfolio_value'] = clients_df['portfolio_value'].apply(lambda x: float(f"{x:.2f}"))


# Investment Data
num_investments = 100 # More investments than clients
investment_data = []
fund_names = ['Equity Growth', 'Bond Stabilizer', 'Tech Innovators', 'Global Diversified', 'Real Estate Income', 'Emerging Markets']

for i in range(1, num_investments + 1):
    client_id = np.random.randint(1, num_clients + 1)
    fund_name = np.random.choice(fund_names)
    amount_invested = np.random.uniform(1000, 500000)

    # Generate random date within the last 5 years
    start_date = datetime.now() - timedelta(days=5*365)
    random_days = np.random.randint(0, 5*365)
    investment_date = (start_date + timedelta(days=random_days)).strftime('%Y-%m-%d')

    investment_data.append({
        'investment_id': i,
        'client_id': client_id,
        'fund_name': fund_name,
        'amount_invested': amount_invested,
        'date': investment_date
    })

investments_df = pd.DataFrame(investment_data)

# Insert data into tables
clients_df.to_sql('clients', conn, if_exists='replace', index=False)
investments_df.to_sql('investments', conn, if_exists='replace', index=False)

# Commit changes and close connection
conn.commit()
conn.close()

print("Database 'financial_data.db' created and populated with sample data.")

# Display a preview of the tables to verify
conn = sqlite3.connect('financial_data.db')
print("\nClients Table Head:")
print(pd.read_sql_query("SELECT * FROM clients LIMIT 5;", conn))
print("\nInvestments Table Head:")
print(pd.read_sql_query("SELECT * FROM investments LIMIT 5;", conn))
conn.close()

Database 'financial_data.db' created and populated with sample data.

Clients Table Head:
   client_id      name  age risk_profile  portfolio_value
0          1  Client 1   61       Medium       4309665.59
1          2  Client 2   26          Low       1285058.41
2          3  Client 3   49          Low        330358.92
3          4  Client 4   46       Medium       4966080.59
4          5  Client 5   66       Medium       4655851.43

Investments Table Head:
   investment_id  client_id           fund_name  amount_invested        date
0              1         11  Global Diversified    375389.056216  2024-04-26
1              2         11     Bond Stabilizer    364733.510949  2020-11-04
2              3         35     Bond Stabilizer    351462.258734  2024-11-15
3              4         32     Tech Innovators    124798.389700  2021-08-25
4              5          1     Tech Innovators    174690.100876  2024-04-28


Set up GOOGLE_API_KEY

In [ ]:


# Configure Google Generative AI
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
    genai.configure(api_key=GOOGLE_API_KEY)
    print("Google API Key loaded from Colab secrets.")
except Exception as e:
    print(f"Could not load API key from Colab secrets: {e}")
    print("Please ensure 'GOOGLE_API_KEY' is set in Colab secrets or as an environment variable.")



Google API Key loaded from Colab secrets.


Initialize the LLM

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

# Connect to the SQLite database
db = SQLDatabase.from_uri("sqlite:///financial_data.db")

# Create the SQL Database Toolkit
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

# Define a custom prompt template for safety and context
custom_prompt_template = """You are an AI assistant that interacts with a financial database.
Your primary goal is to answer user questions by converting them into SQL queries, executing those queries, and then providing a clear, natural language answer.
Only use the tables provided: clients, investments.
Here are the schemas:
clients: client_id (PK), name, age, risk_profile, portfolio_value
investments: investment_id (PK), client_id (FK), fund_name, amount_invested, date

When responding:
- If the question involves 'portfolio > 10L', interpret '10L' as 1,000,000.
- If the question involves 'portfolio < 5L', interpret '5L' as 500,000.
- Always be concise and directly answer the question.
- If you cannot find relevant information, state that clearly.
- Do not make assumptions or invent data.
- Double-check your SQL query before executing.

Question: {input}
"""

Checking output

In [ ]:



# Create the SQL Agent
agent_executor = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    verbose=True, # Set to True to see the thought process and generated SQL queries
    #agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION, # Specify agent type if needed
    handle_parsing_errors=True, # Helps in robust error handling
    # Adding prompt to the agent creation can be tricky with create_sql_agent directly,
    # often it's baked into the toolkit or agent definition.
    # For a more direct prompt control, you might need to use a custom agent or modify the toolkit's prompt.
    # For now, we rely on the toolkit's inherent prompt and augment with our custom instruction.
)

# Function to query the system
def query_financial_system(question):
    print(f"\nUser Question: {question}")
    try:
        response = agent_executor.invoke({"input": custom_prompt_template.format(input=question)})
        print(f"\nAI Answer: {response['output']}")
    except Exception as e:
        print(f"An error occurred: {e}")

# --- Example Queries ---
query_financial_system("Show me all high-risk clients with portfolio > 10L")



User Question: Show me all high-risk clients with portfolio > 10L


> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables
Action Input:clients, investmentsI should query the schema of the clients table to see what columns I can query.
Action: sql_db_schema
Action Input: clients
CREATE TABLE clients (
	client_id INTEGER, 
	name TEXT, 
	age INTEGER, 
	risk_profile TEXT, 
	portfolio_value REAL
)

/*
3 rows from clients table:
client_id	name	age	risk_profile	portfolio_value
1	Client 1	28	High	2887109.18
2	Client 2	34	High	2394162.26
3	Client 3	47	Medium	3723893.47
*/I should query the clients table for clients with a high risk profile and a portfolio value greater than 1,000,000.
Action: sql_db_query_checker
Action Input: SELECT client_id, name FROM clients WHERE risk_profile = 'High' AND portfolio_value > 1000000 LIMIT 10```sql
SELECT client_id, name FROM clients WHERE risk_profile = 'High' AND portfolio_value > 1000000 LIMIT 10
```Action: sql_db_query
Action Input: SELEC